# EcoSort AI: compare run 1 and run 2 fairly

Run 1 and run 2 were each scored on their own validation photos, so their numbers cannot be compared. This
notebook scores **both models on the same photos**, and only on photos that **neither model trained on**.
It does not train anything and takes about 45 to 60 minutes.

- **Run 1:** `ai-model/weights/best.onnx` in the repo (`ecosort-baseline-2`, trained in Colab).
- **Run 2:** `best.zip` at the top of the repo, which is run 2's `best.pt` (trained on Kaggle, merged in PR #6),
  checked against its SHA-256 and converted to ONNX, the format the app serves.

For each model it reports accuracy (precision, recall, mAP, per class) and false detections at the app's own
settings (confidence 0.35): the share of boxes drawn where there is no waste.

## Setup

Same as the training notebook: GPU T4 x2, Internet on, the `ROBOFLOW_API_KEY` secret (and `NTFY_TOPIC`
if you want a notification at the end). Then **Save Version → Save & Run All (Commit)**.
The comparison table is printed at the end and saved as `comparison.json` in the Output tab.

## 1. Check the settings

In [ ]:
import csv
import os
import subprocess
import threading
import time
import urllib.request
from pathlib import Path

WORK = Path("/tmp")  # code and data: not kept after the run
OUT = Path("/kaggle/working")  # kept after the run: the version's Output tab
RUNS = OUT / "runs"
STARTED = time.time()

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    gpu_names = gpu.stdout.strip().replace("\n", ", ") if gpu.returncode == 0 else ""
except FileNotFoundError:
    gpu_names = ""
if not gpu_names:
    raise SystemExit("No GPU. In the notebook settings set Accelerator to 'GPU T4 x2', then run again.")
print("GPU:", gpu_names)

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    os.environ["ROBOFLOW_API_KEY"] = secrets.get_secret("ROBOFLOW_API_KEY").strip()
except Exception:
    raise SystemExit(
        "No ROBOFLOW_API_KEY secret. Add-ons -> Secrets -> Add secret, label it ROBOFLOW_API_KEY, "
        "paste the Private API Key, and tick Attach for this notebook."
    ) from None
try:
    TOPIC = secrets.get_secret("NTFY_TOPIC").strip()
except Exception:
    TOPIC = ""
print("Roboflow key: found")
print("Phone notifications:", f"on (ntfy topic {TOPIC})" if TOPIC else "off")


def notify(message, priority="default"):
    """Send a phone notification through ntfy, if a topic is set. Never stops the run."""
    if not TOPIC:
        return
    try:
        request = urllib.request.Request(
            f"https://ntfy.sh/{TOPIC}",
            data=message.encode(),
            headers={"Title": "EcoSort training (Kaggle)", "Priority": priority},
        )
        urllib.request.urlopen(request, timeout=10)
    except Exception as error:
        print("Notification failed:", error)


def run(step, *command):
    """Run one step as its own process and show its output. If it fails, notify and stop."""
    print(f"\n=== {step} ===", flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in process.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.rstrip()])[-8:]
    if process.wait() != 0:
        hours = (time.time() - STARTED) / 3600
        notify(f"STOPPED at '{step}' after {hours:.1f} h:\n" + "\n".join(tail[-5:]), priority="urgent")
        raise SystemExit(f"'{step}' failed. Read the output above.")


notify("Kaggle run started")

## 2. Get the code and libraries

In [ ]:
BRANCH = "main"
REPO_URL = "https://github.com/TheAnakin01/ecosort-ai.git"

if not (WORK / "ecosort-ai").exists():
    run("Get the code", "git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORK / "ecosort-ai"))
os.chdir(WORK / "ecosort-ai" / "ai-model")
run("Checkout branch", "git", "checkout", BRANCH)
run("Pull branch", "git", "pull", "origin", BRANCH)
run("Install libraries", "pip", "install", "-q", "-r", "requirements.txt")
run("Code version", "git", "log", "-1", "--format=%h %s")
run("Install ONNX tools", "pip", "install", "-q", "onnx", "onnxslim", "onnxruntime~=1.32.0")


In [ ]:
import subprocess

EXPECTED_BRANCH = "main"


def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        msg = "git " + " ".join(args) + " failed: " + result.stderr
        raise SystemExit(msg)
    return result.stdout.strip()


current_branch = git("branch", "--show-current")
commit_hash = git("rev-parse", "HEAD")
commit_subject = git("log", "-1", "--oneline")

print("=== ECOSORT AI GIT VERIFICATION ===")
print(f"Branch: {current_branch}")
print(f"Commit: {commit_hash}")
print(f"Subject: {commit_subject}")

if current_branch != EXPECTED_BRANCH:
    msg = "Expected branch " + EXPECTED_BRANCH + ", but got " + current_branch + ". Check the notebook's Git clone settings."
    raise SystemExit(msg)


## 3. Download the four Roboflow datasets

In [ ]:
run("Download datasets", "python", "scripts/download_datasets.py")

## 4. Build run 2's dataset, and run 1's split

The first build is exactly run 2's dataset. The second rebuilds run 1's three-source split, only to find
which photos run 1 trained on. Both are deterministic, so they match the original runs.

In [ ]:
run("Build the dataset", "python", "scripts/prepare_dataset.py", *"downloads/trash-detection downloads/trashnet downloads/neat-now downloads/garbage-cls-3".split(), "--resplit", "--max-share", "organic=0.3")
run("Check the dataset", "python", "scripts/validate_dataset.py")

In [ ]:
run(
    "Rebuild run 1's split",
    "python", "scripts/prepare_dataset.py",
    "downloads/trash-detection", "downloads/trashnet", "downloads/neat-now",
    "--resplit", "--out", "datasets/run1-split",
)
run(
    "Photos neither model trained on",
    "python", "scripts/make_comparison_split.py",
    "--val-from", "datasets/ecosort", "--exclude-train-of", "datasets/run1-split", "--out", "datasets/compare",
)

## 5. Get both models

Run 2's file is checked against the SHA-256 of the file that was uploaded, so a changed or broken download stops
the notebook. It is then converted to ONNX, as run 1 already is, so both are scored the way the app runs them.

In [ ]:
import hashlib
import shutil

MODELS = OUT / "models"
MODELS.mkdir(parents=True, exist_ok=True)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


RUN2_URL = "https://raw.githubusercontent.com/TheAnakin01/ecosort-ai/main/best.zip"
RUN2_SHA256 = "e57748e9bdcab9d4d41bda3c0e6b502d5dada0946dc0403c3abf1f74491f6669"
RUN1_ONNX_SHA256 = "0e38fc74c6f5f6d1cc1fb67314191f4a4124fba4874b4310c24375177e7dc687"

run2_pt = MODELS / "run2.pt"
urllib.request.urlretrieve(RUN2_URL, run2_pt)
if sha256(run2_pt) != RUN2_SHA256:
    notify("STOPPED: run 2's model file is not the one that was checked.", priority="urgent")
    raise SystemExit("Run 2's file does not match the expected SHA-256. Do not compare with it.")
run("Convert run 2 to ONNX", "python", "scripts/export_onnx.py", "--weights", str(run2_pt))

run1_onnx = MODELS / "run1.onnx"
shutil.copy2("weights/best.onnx", run1_onnx)
if sha256(run1_onnx) != RUN1_ONNX_SHA256:
    raise SystemExit("weights/best.onnx is not run 1's model any more. Update RUN1_ONNX_SHA256 only if that is intended.")

COMPARE = {
    "run1": ("Run 1 (Colab, ecosort-baseline-2)", run1_onnx),
    "run2": ("Run 2 (Kaggle)", MODELS / "run2.onnx"),
}
print("Both models ready.")

## 6. Score both models on the same photos

In [ ]:
import json

DATA = "datasets/compare/data.yaml"
results = {}
for key, (label, weights) in COMPARE.items():
    run(f"Accuracy: {label}", "python", "scripts/evaluate.py", "--weights", str(weights), "--data", DATA,
        "--split", "val", "--out", str(OUT / f"{key}_val_metrics.json"))
    run(f"False detections: {label}", "python", "scripts/check_hallucinations.py", "--weights", str(weights),
        "--data", DATA, "--split", "val", "--examples", "12", "--out", str(OUT / f"{key}_false_detections"))
    results[key] = {
        "label": label,
        "accuracy": json.loads((OUT / f"{key}_val_metrics.json").read_text()),
        "false_detections": json.loads((OUT / f"{key}_false_detections" / "report.json").read_text()),
    }

## 7. The comparison

Higher is better for precision, recall and mAP. Lower is better for **boxes on nothing**: that is the share of
the model's boxes, at the app's settings, where there is no labelled item at all.

In [ ]:
split = json.loads(Path("datasets/compare/comparison_split.json").read_text())
print(f"Scored on {split['kept']} validation photos that neither model trained on "
      f"({split['left_out_seen_in_training']} left out because run 1 trained on them).\n")

a, b = results["run1"], results["run2"]


def row(name, x, y, lower_is_better=False, percent=False):
    better = "=" if abs(x - y) < 0.005 else ("run 2" if (y < x if lower_is_better else y > x) else "run 1")
    fmt = (lambda v: f"{v:.1%}") if percent else (lambda v: f"{v:.3f}")
    print(f"  {name:28s} {fmt(x):>8s} {fmt(y):>8s}   better: {better}")


print(f"  {'':28s} {'run 1':>8s} {'run 2':>8s}")
for metric, name in (("precision", "Precision"), ("recall", "Recall"), ("map50", "mAP@50"), ("map50_95", "mAP@50-95")):
    row(name, a["accuracy"]["overall"][metric], b["accuracy"]["overall"][metric])
row("Boxes on nothing (all)", a["false_detections"]["share_on_nothing"] or 0,
    b["false_detections"]["share_on_nothing"] or 0, lower_is_better=True, percent=True)

print("\nPer class: AP@50, recall, and boxes on nothing")
per_a = {r["class"]: r for r in a["accuracy"]["per_class"]}
per_b = {r["class"]: r for r in b["accuracy"]["per_class"]}
for cls in ("paper", "plastic", "metal", "organic"):
    if cls in per_a and cls in per_b:
        row(f"{cls} AP@50", per_a[cls]["ap50"], per_b[cls]["ap50"])
        row(f"{cls} recall", per_a[cls]["recall"], per_b[cls]["recall"])
    fa = a["false_detections"]["per_class"][cls]["share_on_nothing"]
    fb = b["false_detections"]["per_class"][cls]["share_on_nothing"]
    if fa is not None and fb is not None:
        row(f"{cls} boxes on nothing", fa, fb, lower_is_better=True, percent=True)

(OUT / "comparison.json").write_text(json.dumps({"photos": split, "results": results}, indent=2) + "\n")
notify(
    "Comparison done. mAP@50-95 run 1 {:.3f} vs run 2 {:.3f}; boxes on nothing {:.1%} vs {:.1%}.".format(
        a["accuracy"]["overall"]["map50_95"], b["accuracy"]["overall"]["map50_95"],
        a["false_detections"]["share_on_nothing"] or 0, b["false_detections"]["share_on_nothing"] or 0,
    ),
    priority="high",
)
print("\nSaved: comparison.json, run1_*/run2_* reports, and the worst photos for each model, in the Output tab.")